In [1]:
import optuna
import re
import numpy as np
from ultralytics import YOLO
import time
import wandb

wandb.init(project="Optuna", name=f"Study_2")

BASE_MODEL = "../yolo26s-pose.pt"
DATASET = "../datasets/Lepidoptera/yolo-config.yaml"


c:\Users\tombe\Documents\_MLE\CV-for-GRIT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me

In [2]:

def attention(opt, momen, lr):
        model = YOLO(BASE_MODEL)
        model.train(data=DATASET, epochs=1, optimizer=opt, momentum=momen, lr0=lr, name="./storage/exp")
        metrics = model.val()
        return metrics.mean_results()[-1]

def objective(trial):
        opt = trial.suggest_categorical("Optimizer", ['SGD', 'Adam', 'AdamW'])
        momen = trial.suggest_float('Momentum', 0.85, 0.99)
        lr = trial.suggest_float('Learning Rate', 0.001, 0.1)
        error = attention(opt, momen, lr)
        return error

In [3]:

start = time.time()

study2 = optuna.create_study(direction="maximize")
study2.optimize(objective, n_trials=50)
study2.best_params


[I 2026-06-24 16:23:06,523] A new study created in memory with name: no-name-88c07004-f6d4-4576-aa6a-430a77e815f0


New https://pypi.org/project/ultralytics/8.4.76 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.14.0 torch-2.11.0+cpu CPU (Intel Core i7-1065G7 1.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/Lepidoptera/yolo-config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.05334017699127805, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../yolo26s-pose.pt, momentum=0.982776

[W 2026-06-24 16:23:07,977] Trial 0 failed with parameters: {'Optimizer': 'AdamW', 'Momentum': 0.9827761354283162, 'Learning Rate': 0.05334017699127805} because of the following error: RuntimeError("Dataset '../datasets/Lepidoptera/yolo-config.yaml' error  Dataset '../datasets/Lepidoptera/yolo-config.yaml' images not found, missing path 'C:\\Users\\tombe\\Documents\\_MLE\\CV-for-GRIT\\models\\datasets\\models\\datasets\\Lepidoptera\\images\\val'\nNote dataset download directory is 'C:\\Users\\tombe\\Documents\\_MLE\\CV-FOR-GRIT\\models\\datasets'. You can update this in 'C:\\Users\\tombe\\AppData\\Roaming\\Ultralytics\\settings.json'").
Traceback (most recent call last):
  File "c:\Users\tombe\Documents\_MLE\CV-for-GRIT\.venv\Lib\site-packages\ultralytics\engine\trainer.py", line 714, in get_dataset
    data = check_det_dataset(self.args.data)
  File "c:\Users\tombe\Documents\_MLE\CV-for-GRIT\.venv\Lib\site-packages\ultralytics\data\utils.py", line 531, in check_det_dataset
    raise F

RuntimeError: Dataset '../datasets/Lepidoptera/yolo-config.yaml' error  Dataset '../datasets/Lepidoptera/yolo-config.yaml' images not found, missing path 'C:\Users\tombe\Documents\_MLE\CV-for-GRIT\models\datasets\models\datasets\Lepidoptera\images\val'
Note dataset download directory is 'C:\Users\tombe\Documents\_MLE\CV-FOR-GRIT\models\datasets'. You can update this in 'C:\Users\tombe\AppData\Roaming\Ultralytics\settings.json'

In [ ]:

fig = optuna.visualization.plot_contour(study2)
fig.show()
fig.write_image("Optuna/exp2a.pdf", scale=1)

fig = optuna.visualization.plot_param_importances(study2)
fig.show()
fig.write_image("Optuna/exp2b.pdf", scale=1)

fig = optuna.visualization.plot_edf(study2)
fig.show()
fig.write_image("Optuna/exp2c.pdf", scale=1)

fig = optuna.visualization.plot_parallel_coordinate(study2)
fig.show()
fig.write_image("Optuna/exp2d.pdf", scale=1)

fig = optuna.visualization.plot_optimization_history(study2)
fig.show()
fig.write_image("Optuna/exp2e.pdf", scale=1)

import joblib
joblib.dump(study2, "study_rfi_2.pkl")

end = time.time()
time_s = end - start
print(f'Time (s): {time_s} seconds')